# Lab 3 – Developing a Tiny Transformer Language Model
PUSL3205 – Large Language Models and Generative AI

In [1]:
%pip install torch datasets

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math, time
import matplotlib.pyplot as plt
from datasets import load_dataset

device = "mps" if torch.backends.mps.is_available() else "cpu"
print("Using device:", device)
torch.manual_seed(42)

Using device: mps


## Task 1 – Load the Dataset

In [ ]:
ds = load_dataset("roneneldan/TinyStories", split="train", streaming=True)
stories = [row["text"] for row in ds.take(5000)]
print("Number of stories loaded:", len(stories))

In [ ]:
for i in range(3):
    print(f"--- Story {i+1} ---")
    print(stories[i])
    print()

In [ ]:
lengths = [len(s) for s in stories]
print("Average length (characters):", int(sum(lengths) / len(lengths)))
print("Shortest:", min(lengths), " Longest:", max(lengths))

plt.hist(lengths, bins=50)
plt.xlabel("Story length (characters)")
plt.ylabel("Number of stories")
plt.title("Distribution of story lengths")
plt.show()

**Observation:** TinyStories contains short, simple children's stories written with a small vocabulary.
The 5,000 stories loaded have an average length of 835 characters (shortest 253, longest 4,123). The histogram shows most stories are between 500 and 1,200 characters, with only a few long ones.
Most stories start with "Once upon a time" and follow a simple structure (characters → problem → happy ending), which makes this dataset suitable for training a small language model.

## Task 2 – Preprocess the Text (Character-level Tokenization)

In [ ]:
text = "\n\n".join(stories)
print("Total characters:", len(text))

chars = sorted(set(text))
vocab_size = len(chars)
print("Vocabulary size:", vocab_size)
print("".join(chars))

In [ ]:
stoi = {ch: i for i, ch in enumerate(chars)}   
itos = {i: ch for i, ch in enumerate(chars)}   

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join(itos[i] for i in l)

print(encode("Once upon a time"))
print(decode(encode("Once upon a time")))

**Explanation:** A character level tokenizer was used, where each unique character is mapped to an integer ID.
The 5,000 stories contain about 4.19 million characters, with a vocabulary of 90 unique characters.
A character vocabulary is much smaller than a word vocabulary, which keeps the model small, but the model must learn to spell words one character at a time.

## Task 3 – Prepare Training Data (Next-token Prediction)

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print("Train tokens:", len(train_data), " Validation tokens:", len(val_data))

In [ ]:
block_size = 128
batch_size = 32    

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch("train")
print("Input shape:", xb.shape, " Target shape:", yb.shape)
print("Input :", repr(decode(xb[0][:40].tolist())))
print("Target:", repr(decode(yb[0][:40].tolist())))

**Explanation:** For next token prediction, each input sequence x of 128 characters is paired with a target sequence y that is the same text shifted by one position.

## Task 4 – Build the Transformer Architecture


In [ ]:
class Head(nn.Module):
    """One head of masked self-attention."""
    def __init__(self, n_embd, head_size, dropout):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
       
        wei = q @ k.transpose(-2, -1) / math.sqrt(k.shape[-1])
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))   
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        return wei @ v

In [ ]:
class MultiHeadAttention(nn.Module):
    """Several attention heads running in parallel, then combined."""
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([Head(n_embd, head_size, dropout) for _ in range(n_head)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

In [ ]:
class FeedForward(nn.Module):
    """A simple 2-layer neural network applied to each position."""
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
class Block(nn.Module):
    """Transformer block: attention + feed-forward, each with LayerNorm and a residual connection."""
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = MultiHeadAttention(n_embd, n_head, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffwd = FeedForward(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))   
        x = x + self.ffwd(self.ln2(x))   
        return x

In [ ]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, n_embd=128, n_head=4, n_layer=4, dropout=0.1):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, n_embd)   
        self.pos_emb = nn.Embedding(block_size, n_embd)    
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)       

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok = self.token_emb(idx)
        pos = self.pos_emb(torch.arange(T, device=idx.device))
        x = tok + pos
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -block_size:])
            probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

**Explanation of the architecture:**
- **Token embeddings** turn each character ID into a 128 dimensional vector.
- **Positional embeddings** add information about where each character is in the sequence, since attention itself has no sense of order.
- **Masked multi-head self-attention** lets each character look at all previous characters (never future ones) to decide what is relevant. 4 heads learn different kinds of relationships in parallel.
- **Feed-forward layers** process each position after attention.
- **Residual connections** (x = x + ...) help gradients flow through the network and make training more stable.
- **Layer normalization** keeps the values in a stable range during training.
- The final linear layer outputs a score for each of the vocabulary characters to predict the next character.

## Task 5 – Initialize the Model

In [ ]:
torch.manual_seed(42)
model = TinyGPT(vocab_size, n_embd=128, n_head=4, n_layer=4, dropout=0.1).to(device)

num_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {num_params:,}")

learning_rate = 3e-4
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
print("Loss function: Cross-entropy (inside model.forward)")
print("Optimizer: AdamW, learning rate =", learning_rate)

**Explanation:** The model has 4 Transformer layers, 4 attention heads and 128 dimensional embeddings.
PyTorch initializes the weights randomly. **Cross entropy loss** measures how wrong the predicted next character probabilities are, and the **AdamW optimizer** updates the weights to reduce that loss.

## Task 6 – Train the Model

In [ ]:
@torch.no_grad()
def estimate_loss(model, eval_iters=50):
    model.eval()
    out = {}
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y = get_batch(split)
            _, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

In [ ]:
def train_model(model, optimizer, max_iters=3000, eval_interval=300):
    history = {"step": [], "train": [], "val": []}
    start = time.time()
    for step in range(max_iters + 1):
        # every eval_interval steps, record train and validation loss
        if step % eval_interval == 0:
            losses = estimate_loss(model)
            history["step"].append(step)
            history["train"].append(losses["train"])
            history["val"].append(losses["val"])
            print(f"step {step:5d} | train loss {losses['train']:.4f} | val loss {losses['val']:.4f} | {time.time()-start:.0f}s")

        xb, yb = get_batch("train")          
        logits, loss = model(xb, yb)        
        optimizer.zero_grad()                
        loss.backward()                      
        optimizer.step()                    
    return history

In [ ]:
history = train_model(model, optimizer, max_iters=3000, eval_interval=300)

## Task 7 – Monitor Training (Loss Curves)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history["step"], history["train"], marker="o", label="Train loss")
plt.plot(history["step"], history["val"], marker="o", label="Validation loss")
plt.xlabel("Training step")
plt.ylabel("Cross-entropy loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

**Observation:** Both losses drop sharply in the first 300 steps (4.59 → 2.31) as the model quickly learns the most common characters, spaces and punctuation. After that, the loss decreases more slowly as it learns words and sentence patterns.
After 3,000 steps the train loss is 1.28 and the validation loss is 1.31. The two curves stay very close together, which shows the model is not overfitting. The loss is still decreasing at the end, so more training would likely improve the model further.

## Task 8 – Evaluate the Model (Validation Loss & Perplexity)

In [ ]:
final = estimate_loss(model, eval_iters=200)
val_loss = final["val"]
perplexity = math.exp(val_loss)

print(f"Final validation loss: {val_loss:.4f}")
print(f"Perplexity: {perplexity:.2f}")

**Explanation:** Perplexity = e^(validation loss). It can be understood as "how many characters the model is choosing between" at each step.
A random model would have a perplexity of 90 . Our model has a final validation loss of 1.3043 and a perplexity of 3.69, meaning that on average it is only as uncertain as choosing between about 3–4 characters instead of 90.
This shows the model has learned a lot about the structure of English text.

## Task 9 – Generate Stories

In [ ]:
def generate_story(prompt, max_new_tokens=500, temperature=0.8):
    model.eval()
    context = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    out = model.generate(context, max_new_tokens=max_new_tokens, temperature=temperature)
    return decode(out[0].tolist())

In [ ]:
torch.manual_seed(1)
print(generate_story("Once upon a time"))

In [ ]:
prompts = ["Once upon a time, there was a little dog", "Tom and Lily went to the park", "One day, a big red ball"]

for p in prompts:
    print("PROMPT:", p)
    print(generate_story(p, max_new_tokens=300))
    print("-" * 80)

In [ ]:
for t in [0.5, 1.0, 1.5]:
    torch.manual_seed(1)
    print(f"=== Temperature {t} ===")
    print(generate_story("Once upon a time", max_new_tokens=250, temperature=t))
    print()

**Effect of temperature:**
- **0.5:** safer, more common words, but somewhat repetitive.
- **1.0:** a good balance; the most story like output.
- **1.5:** much more random, with many made-up words and strange symbols (e.g. "â€™"). These come from the encoding errors in the dataset, because high temperature gives rare characters a higher chance of being chosen.

## Task 10 – Experiment with Model Settings

In [ ]:
experiments = [
    {"name": "Smaller (2 layers, 2 heads)",   "n_layer": 2, "n_head": 2, "iters": 3000},
    {"name": "Deeper (6 layers, 4 heads)",    "n_layer": 6, "n_head": 4, "iters": 3000},
    {"name": "More heads (4 layers, 8 heads)", "n_layer": 4, "n_head": 8, "iters": 3000},
    {"name": "Longer training (6000 steps)",  "n_layer": 4, "n_head": 4, "iters": 6000},
]

results = [{
    "Experiment": "Baseline (4 layers, 4 heads, 3000 steps)",
    "Parameters": num_params,
    "Val loss": round(val_loss, 4),
    "Perplexity": round(perplexity, 2),
    "Time (s)": "~100",
}]
exp_histories = {"Baseline": history}
samples = {}

for exp in experiments:
    print(f"\n===== {exp['name']} =====")
    torch.manual_seed(42)
    m = TinyGPT(vocab_size, n_embd=128, n_head=exp["n_head"], n_layer=exp["n_layer"], dropout=0.1).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=3e-4)

    start = time.time()
    h = train_model(m, opt, max_iters=exp["iters"], eval_interval=1000)
    elapsed = time.time() - start

    final_val = estimate_loss(m, eval_iters=200)["val"]
    results.append({
        "Experiment": exp["name"],
        "Parameters": sum(p.numel() for p in m.parameters()),
        "Val loss": round(final_val, 4),
        "Perplexity": round(math.exp(final_val), 2),
        "Time (s)": round(elapsed),
    })
    exp_histories[exp["name"]] = h

    torch.manual_seed(1)
    ctx = torch.tensor([encode("Once upon a time")], device=device)
    samples[exp["name"]] = decode(m.generate(ctx, 200, temperature=0.8)[0].tolist())

In [ ]:
import pandas as pd
results_df = pd.DataFrame(results)
results_df

In [ ]:
plt.figure(figsize=(9, 5))
for name, h in exp_histories.items():
    plt.plot(h["step"], h["val"], marker="o", label=name)
plt.xlabel("Training step")
plt.ylabel("Validation loss")
plt.title("Validation Loss for Different Model Settings")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
for name, text in samples.items():
    print( name )
    print(text)
    print()

**Generated text comparison:** The quality of the text follows the validation loss.
- The **smaller model** produced the most madeup words , showing it has not learned spelling well.
- The **deeper model** and the **8head model** produced more real words and dialogue , but sentences still do not make sense.
- **Longer training** produced the most readable and story like text: "Once upon a time there was a fall named Max. One day, there little boy and her for the dog" with correct spelling, names, punctuation and dialogue.
Lower validation loss  more realistic text.

## Task 11 – Save the Trained Model

In [ ]:
config = {"vocab_size": vocab_size, "n_embd": 128, "n_head": 4, "n_layer": 4,
          "dropout": 0.1, "block_size": block_size}

torch.save({
    "model_state": model.state_dict(),
    "config": config,
    "stoi": stoi,
    "itos": itos,
}, "tiny_transformer.pt")

print("Model saved to tiny_transformer.pt")

In [ ]:
checkpoint = torch.load("tiny_transformer.pt", map_location=device)
cfg = checkpoint["config"]

loaded_model = TinyGPT(cfg["vocab_size"], n_embd=cfg["n_embd"], n_head=cfg["n_head"],
                       n_layer=cfg["n_layer"], dropout=cfg["dropout"]).to(device)
loaded_model.load_state_dict(checkpoint["model_state"])
loaded_model.eval()

torch.manual_seed(1)
ctx = torch.tensor([encode("Once upon a time")], device=device)
print(decode(loaded_model.generate(ctx, 200, temperature=0.8)[0].tolist()))

## Discussion – Mechanisms Used to Improve Performance
The model uses **masked multi-head attention** to learn from earlier characters, **positional embeddings** to know character order, and **residual connections** with **layer normalization** to keep training stable. During training, **dropout** reduces overfitting and the **AdamW optimizer** updates the weights efficiently. **Temperature** controls how creative the generated text is. In the experiments, **training longer** and **adding more layers** gave the biggest improvements.